# ANSER Brain — fine-tune v3 trên Colab

**GPU đích: L4 22.5GB (Colab Pro).** Notebook chạy tuần tự từ trên xuống.

| Giai đoạn | Nội dung | Thời gian ước tính |
|---|---|---|
| 1 | Cài đặt, sinh dữ liệu, preflight | ~25 phút (phần lớn là gọi DeepSeek API) |
| 2 | Train QLoRA + gộp + lượng tử hoá AWQ | ~2–3 giờ |
| — | **RESTART RUNTIME** (vLLM xung đột thư viện train) | 1 phút |
| 3 | Benchmark model gốc vs model đã train | ~40 phút |

**Nguyên tắc:** không có cell nào tự ý ghi đè dữ liệu cũ. Mọi bước sinh dữ liệu
đều *resume* — chạy lại là tiếp tục chỗ dở, không làm lại từ đầu.

---
## Giai đoạn 1 — Chuẩn bị

### 1.1 Kiểm tra GPU
Nếu không thấy L4: *Runtime → Change runtime type → L4 GPU*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch

print("bf16 hỗ trợ:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else "KHÔNG CÓ GPU")

### 1.2 Gắn Drive + lấy mã nguồn

Lấy mã **thẳng từ GitHub**, không chép từ Drive.

> Bản trên Drive chỉ là ảnh chụp lúc upload. Sửa mã rồi quên upload lại là chạy
> nhầm bản cũ mà **không có gì báo** — đã dính đúng bẫy đó một lần. Cell này in ra
> commit đang chạy; nếu cần hỏi gì về lỗi, gửi kèm dòng `Commit` đó.

Drive vẫn được gắn vì dữ liệu sinh và model lưu ở đó. **Phải chạy lại cell này sau
mỗi lần Restart runtime** — `ANSER_GENERATED_DIR` mất theo mỗi lần restart.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

from google.colab import drive

REPO   = "https://github.com/PCBoiz/ANSER_AI.git"
BRANCH = "feat/workflow-format-and-deterministic-core"   # đổi thành "main" sau khi gộp
WORK   = Path("/content/ANSER_AI")

drive.mount("/content/drive")

# Lấy mã THẲNG TỪ GITHUB. Trước đây cell này chép từ Drive, mà bản trên Drive
# chỉ là ảnh chụp lúc upload — sửa mã rồi quên upload lại là chạy nhầm bản cũ
# mà KHÔNG có gì báo. Đã dính đúng bẫy đó một lần (30/07/2026).
if WORK.exists():
    shutil.rmtree(WORK)

clone = subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(WORK)],
    capture_output=True, text=True,
)
if clone.returncode != 0:
    raise SystemExit(
        f"Không lấy được mã từ GitHub:\n{clone.stderr}\n"
        f'Kiểm tra nhánh "{BRANCH}" còn tồn tại không.\n\n'
        "KHÔNG tự lùi về bản trên Drive: bản đó có thể cũ, và chạy nhầm mã cũ "
        "là loại lỗi không có gì báo."
    )

os.chdir(WORK)
sha = subprocess.run(["git", "log", "-1", "--format=%h  %s"],
                     capture_output=True, text=True).stdout.strip()
print(f"Mã nguồn : {BRANCH}")
print(f"Commit   : {sha}")

# Dữ liệu sinh ghi THẲNG lên Drive — tốn tiền API, không được mất khi hết phiên
GEN_DIR = Path("/content/drive/MyDrive/ANSER_AI_Logistics/generated")
GEN_DIR.mkdir(parents=True, exist_ok=True)
os.environ["ANSER_GENERATED_DIR"] = str(GEN_DIR)
print(f"Dữ liệu sinh: {GEN_DIR}  (giữ qua các phiên, chạy lại là resume)")

### 1.3 Cài thư viện train

Version **ghim cứng** trong `requirements_training.txt`. Đừng cài bản mới nhất:
`train_v2.py` cũ chết chính vì API thư viện trôi theo thời gian.

Colab có thể báo xung đột với gói cài sẵn — bỏ qua được, miễn là import ở cell
sau chạy trót lọt.

In [ ]:
!pip install -q -r offline_training/requirements_training.txt
print("\n--- kiểm tra import ---")
import accelerate
import bitsandbytes
import datasets
import peft
import transformers

for m in (transformers, peft, datasets, accelerate, bitsandbytes):
    print(f"{m.__name__:16s} {m.__version__}")

### 1.4 Kiểm tra dung lượng Drive

Model AWQ đầu ra ~5.5GB lưu vào `ANSER_AI_Logistics/anser-v3-awq`.
Bản gộp fp16 (~16GB) chỉ nằm ở `/content` — **không** đẩy lên Drive.

Thiếu chỗ thì xoá: `_backups` (234MB, bản sao lưu cục bộ không cần trên Drive),
`anser-qwen-lora` và `anser-qwen-distill-awq` (model **v1**, đã bị v2 thay thế).

In [ ]:
import shutil
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/ANSER_AI_Logistics')
DATA_DIR    = Path('/content/drive/MyDrive/ANSER_data')   # nơi để model, có từ trước

free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1e9
print(f'Drive còn trống: {free_gb:5.1f} GB   (cần >= 8 GB)\n')

for base in (PROJECT_DIR, DATA_DIR):
    if not base.exists():
        print(f'{base.name}/  (chưa có)')
        continue
    total = sum(f.stat().st_size for f in base.rglob('*') if f.is_file()) / 1e9
    print(f'{base.name}/  — {total:.1f} GB')
    for d in sorted(p for p in base.iterdir() if p.is_dir()):
        size = sum(f.stat().st_size for f in d.rglob('*') if f.is_file()) / 1e9
        flag = '   <- xoá được (bản sao lưu cục bộ)' if d.name == '_backups' else ''
        flag = '   <- model v1, đã bị v2 thay thế' if d.name.startswith('anser-qwen') else flag
        print(f'    {d.name:28s} {size:5.2f} GB{flag}')

if free_gb < 8:
    print('\n*** THIẾU CHỖ — xoá bớt trước khi train ***')

### 1.5 Khoá API DeepSeek

**Không dán khoá thẳng vào notebook** (AGENTS.md R2b). Thêm vào Colab Secrets:
biểu tượng 🔑 bên trái → *Add new secret* → tên `DEEPSEEK_API_KEY` → bật
*Notebook access*.

Chi phí ước tính cho toàn bộ bước sinh dữ liệu: **dưới 1 USD**.

In [ ]:
import os

from google.colab import userdata

try:
    os.environ["DEEPSEEK_API_KEY"] = userdata.get("DEEPSEEK_API_KEY")
    print("✓ Đã nạp DEEPSEEK_API_KEY từ Colab Secrets (không in ra giá trị)")
except Exception as exc:
    print(f"✗ Chưa lấy được khoá: {exc}")
    print("  Bước 1.6b và 1.6c sẽ không chạy được. Xem hướng dẫn ở cell trên.")

### 1.6 Sinh dữ liệu

Năm bước. **a**, **e** không cần mạng; **b**, **c**, **d** gọi DeepSeek.
Cả ba bước gọi API đều *resume* — đứt giữa chừng thì chạy lại chính cell đó,
nó bỏ qua phần đã xong (dữ liệu nằm trên Drive nên không mất tiền lần hai).

| Bước | Nội dung | Nhánh được dạy |
|---|---|---|
| a + b | Trích xuất yêu cầu báo giá, 25% là cặp 2 lượt | `LOGISTICS_EXTRACT` |
| c | Diễn giải số liệu, giải thích xAI, báo cáo từ số engine thật | `DATA` / `EXPLAIN` / `REPORT` |
| d | Vòng agentic: chọn tool, điền tham số, biết hỏi lại khi thiếu | `AGENT` |
| e | Workflow n8n từ template đang chạy thật | `CODER` |

In [ ]:
# a) Ground truth trích xuất (tất định — nhãn đúng tuyệt đối, 25% là cặp 2 lượt)
!python offline_training/make_extraction_seeds.py --n 600 --n-eval 100

In [ ]:
# b) DeepSeek viết tin nhắn cho ground truth có sẵn (~15 phút, resume được)
!python offline_training/reverse_generate.py

In [ ]:
# c) Diễn giải + giải thích (xAI) + báo cáo — số do engine tất định tính,
#    teacher chỉ viết lời (~8 phút)
!python offline_training/make_narration_pairs.py --n 220

In [ ]:
# d) Vòng agentic — câu hỏi CÓ đủ dữ kiện -> gọi tool thật -> trả lời;
#    kèm ca thiếu dữ kiện phải HỎI LẠI thay vì bịa tham số (~10 phút)
!python offline_training/make_agent_traces.py --n 160

In [ ]:
# e) Cặp yêu cầu -> workflow từ template n8n ĐANG CHẠY THẬT
!python offline_training/make_n8n_pairs.py

In [ ]:
# Bộ eval n8n — 34 đề, KHÔNG cần API key và KHÔNG cắt khỏi tập train.
#
# Bản cũ giữ 5 template làm benchmark. n=5 cho khoảng tin cậy ~48%–100%:
# model hoàn hảo và model tung đồng xu ra hai con số không phân biệt được.
# Chấm điểm n8n chạy validate_workflow() trên đầu ra chứ không so đáp án
# mẫu, nên bộ eval chỉ cần ĐỀ BÀI — viết mới được, không phải cắt.
!python -m offline_training.make_n8n_eval


In [ ]:
# e) Gộp tất cả + lọc + quét secret + ghi vân tay catalog
!python offline_training/build_dataset_v3.py

### 1.7 Preflight — chốt chặn trước khi đốt giờ GPU

Kiểm tra: đủ file, đúng cấu trúc vai, **prompt trong data khớp runtime** (P4),
nhãn hợp lệ theo schema, không có secret, độ dài token vừa ngân sách L4, đủ chỗ Drive.

**Thoát mã 1 = có lỗi chặn. Đừng train khi còn lỗi chặn.**

In [ ]:
!python offline_training/preflight_check.py --tokenizer --gpu L4 \
    --drive /content/drive/MyDrive/ANSER_AI_Logistics

### 2.1 Train QLoRA

Mặc định: Qwen3-8B, LoRA r=32, 2 epoch, seq 8192, lô hiệu dụng 16.
Loss **chỉ tính trên phần trả lời**; chọn checkpoint theo `eval_loss` tốt nhất.

**Liger Kernel là bắt buộc với Qwen3.** Từ vựng 151.936 token khiến hàm loss
của transformers dựng tensor `độ_dài × 151.936` rồi **nhân đôi** khi ép fp32 →
OOM *giữa chừng* khi gặp mẫu dài. Liger gộp `lm_head` + cross-entropy thành một
kernel, không bao giờ dựng nguyên tensor đó.

`PYTORCH_CUDA_ALLOC_CONF` phải đặt **trước khi CUDA khởi tạo** — nên nó nằm ở
cell này, và sau khi restart **đừng chạy lại cell 1.1** (cell đó chạm CUDA).

Muốn chắc không OOM trước khi train thật: bỏ chú thích dòng `EPOCHS = "0.05"`,
chạy ~5 phút, thấy trôi thì chú thích lại.

In [ ]:
# Đặt TRƯỚC mọi thao tác CUDA — chống phân mảnh (lần OOM trước kẹt 2,26 GB)
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Cài CÙNG một lệnh: liger kéo theo transformers mới hơn, mà bản đó từ chối
# chạy 4-bit với bitsandbytes cũ. Cài riêng lẻ là gãy ở bước nạp model.
!pip install -q liger-kernel "bitsandbytes>=0.46.1"

os.environ["BASE_MODEL_ID"] = "Qwen/Qwen3-8B"
os.environ["MAX_SEQ_LEN"]   = "8192"      # hạ về "3072" nếu vẫn OOM
os.environ["LORA_R"]        = "32"
os.environ["LR"]            = "1e-4"
os.environ["EPOCHS"]        = "2"
# os.environ["EPOCHS"]      = "0.05"      # <- chạy thử ~5 phút để chắc không OOM
os.environ["OUT_DIR"]       = "/content/checkpoints/anser-v3"
os.environ["LORA_DIR"]      = "/content/checkpoints/anser-v3-lora"

exec(open('/content/ANSER_AI/offline_training/train_v3.py').read())

### 2.2 Sao lưu LoRA lên Drive ngay

LoRA chỉ ~300MB. Sao lưu trước khi làm bước nặng tiếp theo — nếu quantize hỏng
hoặc mất phiên, không phải train lại.

In [ ]:
!cp -r /content/checkpoints/anser-v3-lora /content/drive/MyDrive/ANSER_AI_Logistics/
!du -sh /content/drive/MyDrive/ANSER_AI_Logistics/anser-v3-lora

### 2.3 Gộp LoRA vào weights đầy đủ

LoRA train trên nền 4-bit nhưng phải gộp vào bản fp16/bf16. Bản gộp ~16GB nằm ở
`/content`, **không** lên Drive.

> **Cell này tự tìm LoRA.** `Runtime > Restart` xoá sạch `/content`, nên bản trên
> Drive mới là bản sống sót — cell ưu tiên nó. Chạy được sau restart mà không
> cần chạy lại 2.1.

In [ ]:
# 2.3 — Gop LoRA. Cell nay TU DAT duong dan, khong dua vao cell 2.1.
#
# Runtime > Restart xoa sach /content, nen LORA_DIR mac dinh
# (/content/checkpoints/anser-v3-lora) bien mat sau moi lan restart. Ban tren
# Drive moi la ban song sot -> uu tien no.
import os
from pathlib import Path

CANDIDATES = [
    "/content/drive/MyDrive/ANSER_AI_Logistics/anser-v3-lora",
    "/content/checkpoints/anser-v3-lora",
]
lora = next((p for p in CANDIDATES if Path(p, "adapter_config.json").exists()), None)
assert lora, (
    "Khong thay LoRA o bat ky dau nao:\n  " + "\n  ".join(CANDIDATES) +
    "\n\nNeu Colab da thu hoi may thi phai train lai tu 2.1."
)
print(f"Dung LoRA tai: {lora}")

os.environ["BASE_MODEL_ID"] = "Qwen/Qwen3-8B"
os.environ["LORA_DIR"]      = lora
os.environ["MERGED_DIR"]    = "/content/checkpoints/anser-v3-merged"

!python offline_training/merge_and_quantize.py --stage merge

### 2.4 Lượng tử hoá AWQ 4-bit

Dữ liệu calibration lấy từ chính `train_v3.jsonl` (in-domain — chuẩn hơn
wikitext mặc định cho tiếng Việt + JSON n8n).

**Nếu cell này lỗi cài đặt `autoawq`:** khả năng cao là xung đột version
`transformers`. Cách xử lý: *Runtime → Restart*, chạy lại cell 1.2 (lấy mã),
rồi chạy thẳng cell này (không cần cài lại thư viện train).

In [ ]:
import os

os.environ['AWQ_DIR'] = '/content/drive/MyDrive/ANSER_AI_Logistics/anser-v3-awq'

!pip install -q autoawq==0.2.9
!python offline_training/merge_and_quantize.py --stage quant

In [ ]:
!du -sh /content/drive/MyDrive/ANSER_AI_Logistics/anser-v3-awq
!ls -la /content/drive/MyDrive/ANSER_AI_Logistics/anser-v3-awq | head

---
## ⚠️ RESTART RUNTIME TẠI ĐÂY

vLLM cần bộ thư viện khác với bộ dùng để train. **Runtime → Restart runtime**,
rồi chạy lại **đúng cell 1.2** (gắn Drive + lấy mã nguồn + đặt
`ANSER_GENERATED_DIR`), sau đó tiếp tục từ 3.1.

Không mất gì: dữ liệu sinh nằm trên Drive, model đã lưu Drive, bản gộp fp16 vẫn
ở `/content` (chỉ mất nếu Colab thu hồi máy, khi đó chạy lại từ 2.3).

---
## Giai đoạn 3 — Đo lường

### 3.1 Cài vLLM

In [ ]:
print("=" * 64)
print("  CELL 3.1 — bản 03/08/2026b (GỠ TensorFlow)")
print("  Không thấy dòng này => đang chạy NOTEBOOK CŨ, phải import lại từ GitHub.")
print("=" * 64)

# VÌ SAO PHẢI GỠ TENSORFLOW
#
#     import vllm -> vllm.config -> transformers.image_processing_auto
#       -> transformers/image_transforms.py:47   import tensorflow as tf
#         -> tensorflow/core/framework/attr_value_pb2.py
#           -> ImportError: cannot import name 'runtime_version' from 'google.protobuf'
#
# Cài vLLM hạ protobuf xuống 4.25.9 — CẢ CÂY PHỤ THUỘC của vLLM cần bản đó, nên
# KHÔNG nâng protobuf lên được. Còn TensorFlow 2.20 có sẵn của Colab lại đòi
# >= 5.28. Hai bên không thể cùng đúng, và ta không dùng TF một dòng nào.
#
# Hai lớp, vì lớp một đã từng không đủ:
#   1. USE_TORCH=1  -> transformers không thèm nhìn TF ("Disabling Tensorflow
#      because USE_TORCH is set"). Chỉ ăn nếu đặt TRƯỚC lần import transformers
#      đầu tiên TRONG TIẾN TRÌNH — nên phải Restart runtime rồi chạy cell này
#      trước mọi cell khác của giai đoạn 3.
#   2. Gỡ hẳn gói tensorflow -> kể cả thứ khác có gọi cũng không còn gì để gãy,
#      và không phụ thuộc vào việc ai chạy cell nào trước.
import os

os.environ["USE_TORCH"] = "1"
os.environ["USE_TF"] = "0"

# COLAB ĐÃ LÊN PYTHON 3.13 (23/08/2026) — `vllm==0.8.5` KHÔNG CÀI ĐƯỢC NỮA.
#
#     ERROR: Ignored the following versions that require a different python
#     version: 0.10.0 Requires-Python <3.13,>=3.9
#     ERROR: Could not find a version that satisfies the requirement vllm==0.8.5
#
# 0.10.2 là bản SỚM NHẤT khai `Requires-Python <3.14` — tức bản đầu tiên đỡ
# 3.13 — và nó VẪN còn `GuidedDecodingParams`. Từ 0.12.0 API đó bị gỡ hẳn
# (đổi sang `StructuredOutputsParams`), mà `engine.py` lại bắt `ImportError`
# rồi chạy tiếp, nên nâng quá 0.11.x là guided decoding TẮT ÂM THẦM.
#
# `transformers` không ghim cứng được nữa: 0.10.2 đòi >= 4.55.2. Trần `<5`
# giữ lại lý do cũ — bản 5.x bỏ `all_special_tokens_extended`, thứ vLLM vẫn
# gọi trong `get_cached_tokenizer` (chốt chặn ở cuối cell này soi đúng nó).
!pip install -q "vllm==0.10.2" "transformers>=4.55.2,<5"
# GỠ SAU khi cài: chính lệnh cài ở trên mới là thứ làm TensorFlow hỏng.
!pip uninstall -y -q tensorflow tf-keras || true

import importlib.util

import transformers
from transformers.utils import is_tf_available

print("\ntransformers            ", transformers.__version__)
print("transformers thấy TF    ", is_tf_available(), "   (phải là False)")
print("gói tensorflow còn cài  ", importlib.util.find_spec("tensorflow") is not None)

assert not is_tf_available(), (
    "transformers VẪN thấy TensorFlow — `import vllm` sẽ chết ở protobuf.\n"
    "Nguyên nhân gần như chắc chắn: transformers đã được import trong tiến trình "
    "này TRƯỚC khi cell này chạy.\n"
    "Cách xử lý: Runtime > Restart runtime, chạy lại cell 1.2, rồi chạy NGAY cell này."
)

import vllm

print("vLLM                    ", vllm.__version__)

# vLLM KHÔNG chặn trần trên cho transformers, nên để pip tự chọn là kéo về bản
# 5.x đã bỏ `all_special_tokens_extended` — thuộc tính vLLM vẫn gọi trong
# `get_cached_tokenizer`. Chốt chặn ở đây, đừng để phát hiện sau 20 phút tải
# model.
from transformers import AutoTokenizer

_tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")
assert hasattr(_tok, "all_special_tokens_extended"), (
    "transformers bản này thiếu `all_special_tokens_extended` — vLLM sẽ chết lúc "
    "dựng tokenizer. Cài lại với 'transformers>=4.55.2,<5' rồi Runtime > Restart."
)
print("\n✓ Sẵn sàng: vLLM nạp được, tokenizer hợp, TensorFlow đã tắt")


### 3.1b Kiểm model AWQ trước khi đốt 40 phút đo

Ba thứ đều làm hỏng cả buổi đo, và không thứ nào lộ ra bằng cách nhìn thư mục:

1. **`config.json` thiếu `quantization_config`** → vLLM nạp như fp16 rồi hết VRAM.
2. **Chat template nằm rời ở `chat_template.jinja`** mà bản `transformers` đang
   cài không đọc file rời → `apply_chat_template` nổ, hoặc tệ hơn là **thay bằng
   template mặc định và mọi điểm số đều sai trong khi lệnh vẫn chạy trót lọt**.
   Cell này vá bằng cách nhét template thẳng vào `tokenizer_config.json` — cách
   đó mọi bản `transformers` đều đọc được.
3. **Thiếu file eval** → `benchmark_v3.py` in `⚠ thiếu … bỏ qua` cho cả bốn hạng
   mục rồi kết thúc **không lỗi gate nào**. Một benchmark đo rỗng trông y hệt
   một benchmark đạt.

`ANSER_GENERATED_DIR` mất sau mỗi lần Restart — nếu mục 3 báo đỏ thì chạy lại cell 1.2.


In [ ]:
import json
import os
import shutil
from pathlib import Path

AWQ = Path("/content/drive/MyDrive/ANSER_AI_Logistics/anser-v3-awq")
BASE = "Qwen/Qwen3-8B"
loi = []

# --- 1. config.json phải khai lượng tử hoá ---------------------------------
cfg = json.loads((AWQ / "config.json").read_text())
q = cfg.get("quantization_config")
print("1. quantization_config:", json.dumps(q) if q else "❌ KHÔNG CÓ")
if not q:
    loi.append("config.json thiếu quantization_config — vLLM sẽ nạp như fp16 rồi hết VRAM")
elif q.get("bits") != 4 or q.get("group_size") != 128:
    loi.append(f"tham số AWQ lệch so với lúc quantize: {q}")

# --- 2. đủ shard ------------------------------------------------------------
idx = json.loads((AWQ / "model.safetensors.index.json").read_text())
shards = sorted(set(idx["weight_map"].values()))
thieu = [s for s in shards if not (AWQ / s).exists()]
print(f"2. shard: {len(shards)} file, thiếu {len(thieu)}", thieu or "")
if thieu:
    loi.append(f"thiếu shard {thieu}")

# --- 3. LẤY LẠI TOKENIZER TỪ MODEL GỐC -------------------------------------
# Lượng tử hoá KHÔNG đụng tới tokenizer, nên bản trên HuggingFace mới là bản
# chuẩn. Bộ tokenizer trong thư mục AWQ do transformers v5 ghi ra lúc quantize,
# đọc bằng 4.51.3 thì nổ:
#
#     AttributeError: 'list' object has no attribute 'keys'
#       tokenization_utils_base.py:1190  _set_model_specific_special_tokens
#
# vì v5 ghi `extra_special_tokens` dạng LIST còn 4.51.3 chờ DICT.
#
# Chép đè từ model gốc giải quyết luôn CẢ chuyện chat template: Qwen3-8B giữ
# template BÊN TRONG tokenizer_config.json — chỗ mọi bản transformers đều đọc
# được — chứ không tách ra file `chat_template.jinja` rời như v5 làm.
from huggingface_hub import snapshot_download

TOKENIZER_FILES = [
    "tokenizer.json", "tokenizer_config.json", "vocab.json",
    "merges.txt", "special_tokens_map.json", "added_tokens.json",
]
print(f"3. lấy tokenizer chuẩn từ {BASE}...")
goc = Path(snapshot_download(BASE, allow_patterns=TOKENIZER_FILES))

for ten in TOKENIZER_FILES:
    src = goc / ten
    if src.exists():
        shutil.copy2(src, AWQ / ten)
        print(f"     ✓ {ten}")

# Template rời do v5 để lại: dọn đi cho khỏi hai nguồn sự thật đá nhau.
sidecar = AWQ / "chat_template.jinja"
if sidecar.exists():
    sidecar.rename(AWQ / "chat_template.jinja.bak")
    print("     ✓ chat_template.jinja -> .bak (template đã nằm trong tokenizer_config)")

import transformers
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(str(AWQ))
print(f"   transformers {transformers.__version__} | thấy template:", bool(tok.chat_template))
if not tok.chat_template:
    loi.append("Vẫn KHÔNG có chat template — mọi prompt sẽ sai định dạng, điểm số vô nghĩa")
else:
    s = tok.apply_chat_template(
        [{"role": "user", "content": "xin chào"}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    print("   render đuôi:", repr(s[-90:]))
    # Qwen3 với enable_thinking=False sinh ra một khối <think> RỖNG ngay sau
    # <|im_start|>assistant. Đó là ĐÚNG — khối rỗng chính là tín hiệu
    # 'đừng suy nghĩ' gửi cho model.
    # Bản kiểm đầu bắt lỗi hễ thấy chuỗi "<think>", nên nó gắn cờ đỏ đúng lúc
    # mọi thứ đang chạy chuẩn và chặn cả buổi đo (04/08/2026).
    # Cái đáng lo là khối <think> CÓ NỘI DUNG, hoặc mở mà không đóng.
    import re as _re
    than_think = _re.search(r"<think>(.*?)</think>", s, _re.S)
    if than_think and than_think.group(1).strip():
        loi.append("enable_thinking=False vô tác dụng — <think> có nội dung, "
                   "model sẽ suy luận dài và ăn hết ngân sách token")
    elif "<think>" in s and "</think>" not in s:
        loi.append("template mở <think> mà không đóng — model sẽ suy luận không dừng")

# --- 4. file eval (mất sau mỗi lần Restart runtime) -------------------------
gen = Path(
    os.environ.get("ANSER_GENERATED_DIR", "").strip()
    or "/content/drive/MyDrive/ANSER_AI_Logistics/generated"
)
print("4. GENERATED_DIR:", gen)
for name in ("eval_extraction.jsonl", "eval_n8n.jsonl",
             "eval_narration.jsonl", "eval_agent.jsonl"):
    p = gen / name
    n = sum(1 for _ in p.open(encoding="utf-8")) if p.exists() else 0
    print(f"     {'✓' if p.exists() else '❌'} {name} {f'({n} dòng)' if p.exists() else ''}")
    if not p.exists():
        loi.append(f"thiếu {name} — benchmark BỎ QUA hạng mục này mà không báo lỗi gate")

# --- 5. mã nguồn đang chạy có phải bản mới không ----------------------------
# Cell 1.2 clone từ GitHub. Sửa code ở máy mà quên push là Colab chạy bản CŨ,
# và không có gì báo — đúng cái bẫy đã dính một lần (30/07/2026).
import subprocess

sha = subprocess.run(["git", "-C", "/content/ANSER_AI", "log", "-1", "--format=%h %ci %s"],
                     capture_output=True, text=True).stdout.strip()
print("5. commit đang chạy:", sha or "(không đọc được)")
src = Path("/content/ANSER_AI/offline_training/benchmark_v3.py").read_text(encoding="utf-8")
if 'quantization = "awq" if "awq" in model_path.lower()' in src:
    loi.append(
        "benchmark_v3.py là BẢN CŨ — vẫn ép quantization='awq' nên sẽ nổ "
        "'bfloat16 is not supported for awq'. Push bản vá lên GitHub rồi chạy lại cell 1.2."
    )
# Từ 06/08/2026 tool do BẢNG LUẬT tất định chọn, không phải model. Bản benchmark
# cũ vẫn chấm "model chọn đúng tool" — chỉ tiêu nay vô nghĩa vì enum trong grammar
# chỉ có một tên. Chạy nhầm bản cũ thì ra một con số trông bình thường nhưng đo
# một khâu không còn tồn tại, và đó là kiểu sai tốn nhất: không có gì báo.
if "def score_planner" not in src:
    loi.append(
        "benchmark_v3.py là BẢN TRƯỚC 06/08 — chưa tách phần chấm bảng luật khỏi "
        "phần chấm model. Chạy lại cell 1.2 để clone bản mới từ GitHub."
    )
if "def arguments_schema" not in Path(
    "/content/ANSER_AI/src/agents/agentic.py"
).read_text(encoding="utf-8"):
    loi.append(
        "agentic.py là BẢN CŨ — `arguments` chưa bị chặn, model sẽ viết ra cả "
        "mảng dữ liệu rồi cắt cụt. Chạy lại cell 1.2."
    )

print("\n" + ("❌ CHƯA ĐO ĐƯỢC:\n  - " + "\n  - ".join(loi)
             if loi else "✅ ĐỦ ĐIỀU KIỆN — chạy 3.2 và 3.3"))


### 3.2 Baseline — model GỐC chưa fine-tune

**Chạy cell này trước khi xem điểm model mới.** Không có baseline thì con số
"85% đúng" chẳng nói lên điều gì: có thể model gốc đã đạt 84%, tức fine-tune
gần như vô ích; cũng có thể gốc chỉ 40%, tức cải thiện rất lớn.

`--no-gate` = chỉ đo, không chặn. Mất ~20 phút (phần lớn là tải model 16GB).

In [ ]:
# Baseline — model GỐC. `--json` ghi kết quả TỪNG CÂU để ô 3.4 so theo cặp.
!python offline_training/benchmark_v3.py --model Qwen/Qwen3-8B --no-gate --json /content/baseline.json 2>&1 | tee /content/baseline_report.txt


### 3.3 Model đã fine-tune

Không có `--no-gate` → **thoát mã 1 nếu dưới ngưỡng**. Ngưỡng mặc định:

| Chỉ số | Ngưỡng | Ý nghĩa |
|---|---|---|
| `EXTRACT_FIELD_MIN` | 0.85 | độ chính xác trung bình các trường trích xuất |
| `EXTRACT_READY_MIN` | 0.80 | tỷ lệ đủ 3 trường bắt buộc để tính được báo giá |
| `EXTRACT_FOLLOWUP_MIN` | 0.70 | **riêng câu nối tiếp** — đo khả năng kế thừa ngữ cảnh |
| `N8N_VALID_MIN` | 0.90 | workflow sinh ra qua được validator |
| `NARR_MIN` | 0.90 | không bịa số, không lộ biên lợi nhuận |
| `AGENT_TOOL_MIN` | 0.85 | chọn đúng công cụ cho câu hỏi có đủ dữ kiện |
| `AGENT_ASKBACK_MIN` | 0.75 | **thiếu dữ kiện thì hỏi lại**, không gọi tool với tham số bịa |

Hai chỉ số cuối là mới. `AGENT_ASKBACK_MIN` quan trọng hơn `AGENT_TOOL_MIN`:
chọn nhầm tool thì kết quả sai lộ ra ngay, còn gọi tool với tham số **bịa** thì
ra một con số trông hợp lý mà không ai biết là sai.

In [ ]:
AWQ = "/content/drive/MyDrive/ANSER_AI_Logistics/anser-v3-awq"
!python offline_training/benchmark_v3.py --model {AWQ} --json /content/tuned.json 2>&1 | tee /content/tuned_report.txt


### 3.4 So sánh

Đọc theo thứ tự ưu tiên:

1. **`sẵn sàng báo giá`** — chỉ số sát nghiệp vụ nhất: bao nhiêu % tin nhắn
   khách gửi ra được báo giá mà không phải hỏi lại.
2. **`câu nối tiếp`** — thấp hơn hẳn câu một lượt nghĩa là hội thoại nhiều lượt
   chưa dùng được, dù điểm chung có đẹp.
3. **`false_fill`** — model **đoán bừa** trường mà tin nhắn không nêu. Đây là
   lỗi nguy hiểm nhất: nó phá nhánh hỏi-lại và tạo báo giá sai tuyến trong im lặng.
4. **`n8n hợp lệ`** — dưới 90% thì nhánh tạo quy trình chưa giao cho khách được.

In [ ]:
# So HAI BẢN THEO CẶP trên cùng bộ câu hỏi.
#
# Hai file .txt chỉ cho hai con số trung bình, và số trung bình GIẤU đúng
# thứ nguy hiểm nhất: bản mới đã làm hỏng câu nào mà baseline vốn trả lời
# đúng? 70% -> 70% có thể là "hỏng 3, sửa 3" — ba tình huống khách từng
# dùng được, nay không.
!python -m offline_training.compare_runs /content/baseline.json /content/tuned.json

# Lưu cả báo cáo văn bản lẫn kết quả từng câu lên Drive để đối chiếu lần sau
DRIVE = "/content/drive/MyDrive/ANSER_AI_Logistics/"
!cp /content/baseline_report.txt /content/tuned_report.txt {DRIVE}
!cp /content/baseline.json /content/tuned.json {DRIVE}
!ls -la {DRIVE}*.txt {DRIVE}*.json


---
## Xong. Dùng model như thế nào

Trên máy chạy Brain (không phải Colab — xem ARCHITECTURE §4.4: Colab **cấm**
phục vụ traffic thật):

```bash
export TEXT_MODEL_ID=/đường/dẫn/tới/anser-v3-awq
```

`config.py` đọc biến này. **Không cần khai `quantization`** — vLLM tự đọc
`quantization_config` trong `config.json` của model rồi chọn nhân hợp GPU
(`awq_marlin` từ Ampere trở lên, `awq` trên Turing).

Khai cứng `quantization="awq"` vẫn chạy, nhưng ép xuống nhân chậm hơn **và**
khiến benchmark đo một nhân khác nhân lúc serve — số đo khi đó không nói gì về
lúc chạy thật, mà đó lại là toàn bộ mục đích của việc đo (sửa 03/08/2026).

Ép tay khi cần: `TEXT_QUANTIZATION=awq` **phải** kèm `TEXT_DTYPE=half`. Nhân
`awq` cũ chỉ nhận `float16`; để `auto` là ra `bfloat16` rồi nổ ngay lúc khởi tạo
với `ValueError: torch.bfloat16 is not supported for quantization method awq`.

### Nếu điểm chưa đạt ngưỡng

| Triệu chứng | Hướng xử lý |
|---|---|
| `false_fill` cao (đoán bừa trường thiếu) | Tăng tỷ lệ seed thiếu trường: sửa `MISSING_PATTERNS` trong `make_extraction_seeds.py` |
| Câu nối tiếp kém hơn hẳn câu một lượt | Tăng `FOLLOWUP_RATIO` (mặc định 0.25) rồi sinh lại dữ liệu |
| n8n hợp lệ thấp | Kiểm tra `N8N_TEMPLATES_DIR` lúc serve có trỏ đúng thư mục lúc train không — `preflight_check.py` so vân tay catalog |
| Mọi chỉ số đều kém hơn baseline | Overfit: giảm `EPOCHS` về 1, hoặc giảm `LORA_R` về 16 |
| Điểm gần bằng baseline | Fine-tune đóng góp ít — guided decoding đã gánh phần lớn. Cân nhắc dùng thẳng model gốc, để dành tiền cho hạ tầng |


---
## Giai đoạn 4 — Chạy thử đầu-cuối qua đường hầm

Ba giai đoạn trên đo model **rời**: nạp weights, hỏi, chấm điểm. Giai đoạn này
dựng Brain lên thật rồi cho Body gọi vào, để thấy phần mà benchmark không thấy:
đường xác thực, luồng task bất đồng bộ, độ trễ khi đi qua mạng.

> ⚠️ **Đây là phiên demo rời, KHÔNG phải cách triển khai.** AGENTS.md §3.1:
> điều khoản Colab Paid **cấm phục vụ web service**, và ARCHITECTURE.md §4.4 đã
> bỏ hẳn kiến trúc FastAPI-trên-Colab. Đường triển khai thật là Cloudflare Tunnel
> trong `deploy/docker-compose.yml`. Dùng mục này để **thử nghiệm rời** theo
> §3.1b, rồi tắt đi — đừng đưa URL này cho khách.

**Chạy sau ô 40.** Lúc đó GPU đã được trả lại: ô 36/38 gọi benchmark bằng
`!python`, tiến trình con đã thoát nên weights không còn giữ VRAM.

### Về ô "chạy thử tích hợp" ở cuối

`benchmark_integration.py` có 6 ca. Cần biết trước khi đọc kết quả:

| Ca | Trạng thái |
|---|---|
| T1, T3 | `create_workflow` — **còn đúng**, đây vẫn là hợp đồng n8n hiện tại |
| **T2** | **HỎNG SẴN** — nó soi `{"action": "query_db", "sql": ...}`, hợp đồng đã bị gỡ. Ca này trượt dù model tốt tới đâu |
| T4, T5, T6 | hoá đơn / thuế / chuyển hướng — còn hợp lý, nhưng chưa ai chạy lại từ khi kiến trúc đổi |

Nên **`passed_cases` có sẵn một ca trượt**, và ngưỡng `>= 4` của script đọc theo
nghĩa cũ. Thứ đáng tin ở ô này là **độ trễ p95**, `/health`, và việc luồng
`POST /chat` → `GET /api/v1/task/{id}` chạy thông. Đừng đọc `4/6` thành điểm model.


In [ ]:
# ---- 4.1 Thư viện + khoá ---------------------------------------------------
#
# `requirements.txt` là danh sách của tầng SERVE, khác danh sách lúc train.
# Cài xong phải GHIM LẠI vllm/transformers. `requirements.txt` chặn `<0.12`
# nhưng vẫn để pip chọn trong khoảng, mà ô 32 đã chọn đúng cặp cho Qwen3.
# Quan trọng hơn: 0.12.0 trở lên GỠ `GuidedDecodingParams`, và `engine.py`
# bắt `ImportError` rồi chạy tiếp — Brain sẽ phục vụ với guided decoding TẮT
# ÂM THẦM, chỉ để lại một dòng log.
import os
import secrets

!cd /content/ANSER_AI && pip install -q -r requirements.txt
!pip install -q "vllm==0.10.2" "transformers>=4.55.2,<5"

from google.colab import userdata

# ngrok: tài khoản ẩn danh cũng chạy nhưng hay bị chặn giữa chừng.
# Thêm NGROK_AUTHTOKEN vào Colab Secrets (biểu tượng chìa khoá bên trái).
try:
    os.environ["NGROK_AUTHTOKEN"] = userdata.get("NGROK_AUTHTOKEN")
    print("✓ có NGROK_AUTHTOKEN")
except Exception:
    print("⚠ chưa có NGROK_AUTHTOKEN trong Colab Secrets — sẽ mở hầm ẩn danh")

# ---------------------------------------------------------------------------
# API_AUTH_TOKEN RỖNG = BRAIN KHÔNG KIỂM TOKEN NÀO.
#
# `require_api_token` (src/api/dependencies.py) return sớm khi biến này rỗng.
# Ngày 13/08/2026 đúng lỗi này lọt ra: compose đặt tên biến là `API_TOKEN`
# trong khi code đọc `API_AUTH_TOKEN`, nên một Brain không kiểm token nào bị
# đưa thẳng ra Internet — mà người vận hành thì đinh ninh đã khoá vì compose
# vừa bắt điền token.
#
# Ở đây sinh token ngẫu nhiên mỗi phiên, không bao giờ để rỗng.
#
# `setdefault` CHỨ KHÔNG gán đè — và đây không phải chi tiết văn phong.
# Bản trước gán thẳng, nên chạy lại ô này SAU khi server đã bật là sinh token
# mới trong khi server vẫn giữ token cũ. Hậu quả: mọi request trả 401, mà
# /health vẫn xanh vì nó không đòi token — nhìn y hệt "model chết".
# Gặp thật 23/08/2026.
# ---------------------------------------------------------------------------
os.environ.setdefault("API_AUTH_TOKEN", secrets.token_hex(32))


def dau_van(tok):
    """Vân tay để đối chiếu bằng mắt giữa các ô — không in cả token."""
    return f"{tok[:4]}…{tok[-4:]}"


print("✓ API_AUTH_TOKEN:", dau_van(os.environ["API_AUTH_TOKEN"]))
print("  (chạy lại ô này KHÔNG đổi token — server đang chạy vẫn dùng được)")


In [ ]:
# ---- 4.2 Bật Brain ở nền -----------------------------------------------------
import json
import os
import subprocess
import time
import urllib.error
import urllib.request

AWQ = "/content/drive/MyDrive/ANSER_AI_Logistics/anser-v3-awq"
assert os.path.isdir(AWQ), f"Không thấy {AWQ} — chạy ô 26/28 trước"

# KHÔNG khai TEXT_QUANTIZATION / TEXT_DTYPE. Để vLLM tự đọc `quantization_config`
# rồi chọn nhân hợp GPU — đúng lý do ghi ở ô 41. Ép tay `awq` bắt lúc serve chạy
# NHÂN KHÁC nhân mà ô 36/38 vừa đo, và khi đó số đo không còn nói gì về lúc chạy
# thật, tức là mất luôn mục đích của cả phiên đo.
env = {
    **os.environ,
    "ENV": "SERVER",                       # != LOCAL thì mới nạp model thật
    "TEXT_MODEL_ID": AWQ,
    "PYTHONPATH": "/content/ANSER_AI",
    "VLLM_WORKER_MULTIPROC_METHOD": "spawn",
}

# GIẾT SERVER CŨ TRƯỚC.
#
# Ô 4.5 chỉ gọi `proc.terminate()`. `proc` không tồn tại (chạy lại ô lẻ,
# hoặc đã Restart runtime) thì NameError bị `except Exception` nuốt, và
# uvicorn cũ vẫn giữ cổng 8000. Khi đó `Popen` dưới đây không bind được,
# tiến trình mới chết, còn vòng đợi thì thấy `/health` XANH — vì server CŨ
# trả lời. Ô báo thành công, và mọi request sau đó ăn 401 vì server cũ giữ
# token cũ. Gặp thật 23/08/2026.
!pkill -f "uvicorn src.api.main:app" 2>/dev/null; sleep 2

if doc_health(timeout=2.0) is not None:
    raise SystemExit(
        "Cổng 8000 VẪN có người nghe sau khi pkill. Đừng khởi động chồng lên "
        "— server mới sẽ chết lặng và server cũ tiếp tục trả lời bằng token cũ.\n"
        "Runtime > Restart runtime, rồi chạy lại ô 4 và Giai đoạn 4."
    )

proc = subprocess.Popen(
    ["uvicorn", "src.api.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/ANSER_AI", env=env,
)
print(f"uvicorn pid={proc.pid} — đợi /health…")
print(f"  token server đang giữ: {env['API_AUTH_TOKEN'][:4]}…"
      f"{env['API_AUTH_TOKEN'][-4:]}   (phải khớp vân tay ô 4.1 và ô 4.4)")

# `/health` trả lời NGAY khi uvicorn lên: `lifespan` không nạp gì, model nạp
# LƯỜI ở request đầu tiên (`ensure_text_runtime`). Nên vòng đợi dưới đây chỉ
# mất vài giây — còn 1–3 phút nạp weights sẽ rơi vào CÂU HỎI ĐẦU TIÊN, dù là
# ô 4.4 hay câu bạn gõ trong khung chat của Body. Đừng tưởng là treo.


def doc_health(timeout=3.0):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=timeout) as r:
            return json.loads(r.read())
    except (urllib.error.URLError, TimeoutError, ConnectionError):
        return None


health = None
for _ in range(120):                       # 120 × 5s = 10 phút
    if proc.poll() is not None:
        raise SystemExit(f"uvicorn chết sớm, mã thoát {proc.returncode} — đọc log ở trên")
    health = doc_health()
    if health:
        break
    time.sleep(5)

assert health, "Brain không trả lời /health sau 10 phút — đọc log uvicorn ở trên"
print(json.dumps(health, ensure_ascii=False, indent=1)[:800])

# Chốt chặn: /health có trường `auth_enabled` chính là để nhìn thấy sự cố 13/08.
assert health.get("auth_enabled"), (
    "Brain đang KHÔNG kiểm token. Chạy lại ô 4.1 — `API_AUTH_TOKEN` phải có "
    "giá trị TRƯỚC khi uvicorn khởi động, vì dependencies.py đọc nó lúc import."
)
# XÁC MINH ĐÚNG SERVER. `/health` không đòi token nên nó KHÔNG phân biệt được
# server của mình với server cũ đang giữ cổng 8000.
#
# BODY PHẢI HỢP LỆ. Bản đầu gửi `{}` cho gọn và luôn nhận 422 — vì FastAPI
# validate body TRƯỚC khi chạy thân hàm, nên `require_api_token` KHÔNG BAO GIỜ
# được chạy tới. Chốt chặn đó cho đèn xanh với mọi token, kể cả token bịa. Tự
# dựng một cái kiểm không kiểm gì (23/08/2026).
#
#   401 = token sai  ->  server trả lời KHÔNG phải server vừa bật
#   200 = token đúng
#   422 = lược đồ VatRequest đã đổi, phép thử này cần sửa lại
import urllib.error

req = urllib.request.Request(
    "http://127.0.0.1:8000/tools/vat",
    data=b'{"items":[],"stated_total":0}',
    headers={"Content-Type": "application/json",
             "X-API-Token": env["API_AUTH_TOKEN"]},
    method="POST",
)
try:
    urllib.request.urlopen(req, timeout=10).read()
    ma = 200
except urllib.error.HTTPError as e:
    ma = e.code

if ma == 401:
    proc.terminate()
    raise SystemExit(
        "Server đang nghe cổng 8000 KHÔNG phải server vừa bật: nó từ chối "
        f"token {dau_van(env['API_AUTH_TOKEN'])}.\n"
        "Gần như chắc chắn là một uvicorn cũ chưa chết hẳn.\n"
        "Runtime > Restart runtime, rồi chạy lại ô 4 và Giai đoạn 4."
    )
if ma == 422:
    print("  ⚠ /tools/vat trả 422 — lược đồ VatRequest đã đổi, nên phép thử này\n"
          "    KHÔNG kiểm được auth nữa (body sai thì FastAPI chặn trước auth).\n"
          "    Sửa body trong ô này cho khớp lược đồ mới.")

print(f"\n✓ Brain sống, ĐANG kiểm token, và nhận token của phiên này "
      f"(/tools/vat -> {ma}, khác 401)")


In [ ]:
# ---- 4.3 Mở đường hầm ---------------------------------------------------------
import os

from pyngrok import conf, ngrok

token = os.environ.get("NGROK_AUTHTOKEN", "").strip()
if token:
    ngrok.set_auth_token(token)
conf.get_default().region = "ap"           # gần VN hơn, đỡ một vòng địa cầu

ngrok.kill()                               # dọn hầm cũ nếu ô này chạy lại
BRAIN_URL = ngrok.connect(8000).public_url
os.environ["BRAIN_URL"] = BRAIN_URL

print("=" * 70)
print("  DÁN HAI DÒNG NÀY VÀO  frontend/.env.local  CỦA BODY")
print("=" * 70)
print(f"BRAIN_URL={BRAIN_URL}")
print(f"BRAIN_API_TOKEN={os.environ['API_AUTH_TOKEN']}")
print("=" * 70)
print("\nRồi `npm run dev`, mở /dashboard và hỏi một câu trong khung chat.")
print("Body gửi token qua header `X-API-Token` — đúng tên Brain đang đọc.")
print("\n⚠ URL này sống theo phiên Colab. Ngắt phiên là hỏng, phải dán lại.")


In [ ]:
# ---- 4.4 Chạy thử tích hợp ----------------------------------------------------
#
# ĐỌC PHẦN CẢNH BÁO Ở Ô MARKDOWN ĐẦU GIAI ĐOẠN 4 TRƯỚC KHI ĐỌC BẢNG DƯỚI.
# Tóm lại: T2 soi hợp đồng `query_db` đã bị gỡ nên TRƯỢT SẴN, không phải lỗi
# model. Thứ đáng tin ở đây là độ trễ p95 và việc luồng task async chạy thông.
#
# Script đọc token từ BIẾN MÔI TRƯỜNG, không nhận qua dòng lệnh — để token
# không nằm lại trong output notebook. `!lệnh` của Colab kế thừa os.environ,
# nên `API_AUTH_TOKEN` ô 4.1 đặt sẽ tự tới nơi.
#
# Chạy ô này mà thấy 401 trên CẢ SÁU ca: đó là MỘT lỗi cấu hình, không phải
# sáu ca hỏng. Nghĩa là bản clone trong /content/ANSER_AI còn cũ hơn bản sửa —
# lấy riêng file đó về:
#
#   !cd /content/ANSER_AI && git fetch -q origin feat/lop-provider-cho-benchmark \
#      && git checkout FETCH_HEAD -- offline_training/benchmark_integration.py
!cd /content/ANSER_AI && BRAIN_URL={BRAIN_URL} python offline_training/benchmark_integration.py


In [ ]:
# ---- 4.5 Dọn ------------------------------------------------------------------
#
# Đóng hầm TRƯỚC khi tắt server: để ngược lại thì có một khoảng URL còn sống mà
# phía sau không còn ai trả lời.
from pyngrok import ngrok

ngrok.kill()
print("✓ đã đóng đường hầm")

try:
    proc.terminate()
    proc.wait(timeout=30)
    print("✓ đã tắt uvicorn")
except Exception as exc:
    print(f"⚠ terminate() không gọn: {exc}")

# KHÔNG dựa vào `proc`. Biến đó biến mất sau Restart runtime hoặc khi ô này
# chạy lẻ, và khi đó NameError bị nuốt còn uvicorn thì vẫn sống — giữ cổng
# 8000 để lần bật sau khởi động chồng lên nó. `pkill` theo dòng lệnh là thứ
# đúng dù `proc` có tồn tại hay không.
!pkill -f "uvicorn src.api.main:app" 2>/dev/null; sleep 2

import urllib.request

try:
    urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2).read()
    print("⚠ CỔNG 8000 VẪN CÓ NGƯỜI NGHE — Runtime > Restart runtime")
except Exception:
    print("✓ cổng 8000 đã trống")

!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
